In [1]:
#import the library pulp as p
import pulp as p
from pulp import *
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
from scipy.optimize import minimize

from Interpolation_functions import *

In [2]:
import gurobipy as gp
print(gp.gurobi.version())
model = gp.Model()
print("License ID:", model.getParamInfo("LicenseID"))

(12, 0, 1)
Set parameter Username
Set parameter LicenseID to value 2623371
Academic license - for non-commercial use only - expires 2026-02-18
License ID: ('LicenseID', <class 'int'>, 2623371, 0, 2000000000, 0)


In [3]:
#common parameters for all the experiments/GLOBAL VARIABLES
stock = 1 #initial stock price

K = 1 # target option's strike price

r = 0 #risk-free rate of interest

sigma = 0.2 #volatility of the stock price

T = 1 #target maturity

mu = 0 #return of the stock price

t = 0

timepoints =2 #number of timesteps


T_GRID = [0,0.5,T]#times at which the marginals of the stock price are evaluated

short_maturity = T_GRID[1] # maturity of the options in the hedge portfolio


short_strike_range = [1,1.1,0.9,1.2,0.8,1.25]#range of short strikes for increasing number of options

In [4]:
A=[2,3,4]
print(np.append(A[:-1],2))

[2 3 2]


In [5]:
# #function to return the discretization points, call prices and the marginals for a given value of truncation range and number of discretization points and choice of uniform or nonuniform grid

# def get_disc_points_and_marginals(truncation_range=None,n=None, N=None,choice="uniform"):
#     """
#        Generate the discretization points for the given problem depending on uniform or non-uniform grid
#        Parameters: 
#           Global variables: Stock, sigma, T,r,
#           Local variables: n- (1/n) denotes the spacing between the discretization points
                        
#     """
#     if choice == "uniform":
#         if truncation_range == None or n == None:
#             return ("Need to provide n or truncation range")
#         #return a uniform grid
#         D = np.linspace(0, truncation_range, int(truncation_range*n)+1)
        
        
#         #next we find the point where the interpolated line joining the last two strike points takes value zero
#         R_0_T1 = find_zero(BS_call,stock,D,T_GRID[1],sigma,r)
#         print(f"The strike for time {T_GRID[1]} is {R_0_T1} and interpolated call value: {interpolated_line(stock,D,T_GRID[1],sigma,r,R_0_T1)}")
        
#         R_0_T = find_zero(BS_call,stock,D,T_GRID[1],sigma,r)
#         print(f"The strike for time {T} is {R_0_T} and interpolated call value: {interpolated_line(stock,D,T,sigma,r,R_0_T)}")
        
#         R_0 = max(R_0_T1,R_0_T)
# #         print(f"The interpolated line takes the value 0 at {R_0}")
       
#         #print the corresponding call value
#         print(f"Strike:{R_0},Interpolated call value:{interpolated_line(stock,D,T_GRID[1],sigma,r,R_0)}")
        
        
#         #extending the truncation range to include the new points till the zero of the interpolated call value
#         K1_new = append_numbers(D,D[-1],R_0,np.diff(D)[0])
        
# #         print(f"Extended truncation range for time {t1[1]}:{K1_new}")
# #         print(f"length of new truncation range :{len(K1_new)}")
# #         print(f"previous length:{len(D)}")
        
#         K2_new = append_numbers(D,D[-1],R_0,np.diff(D)[0])
# #         print(f"Extended truncation range for time {t1[2]}:{K2_new}")
# #         print(f"spacing:{K2_new[-2]/(len(K2_new)-2)},{D[-1]/(len(D)-1)}") 
        
        
#         #getting the new discretization points
#         disc_points_new = disc_points_new_1(K1_new,K2_new)
#         print(f"The new discretization points are:{disc_points_new}")
        
#         # store the call prices as a 2D array
#         call_prices_array_new = np.vstack((call_prices(stock,D,K1_new,T_GRID[1],sigma,r),call_prices(stock,D,K2_new,T,sigma,r)))
# #         print(call_prices_array_new)
        
#         #getting the discrete marginal distributions

#         marg = marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID)
# #         print(marginal_new(call_prices_array_new,disc_points_new,timepoints,t1))
        
        
#         #returns the grid and the number of grid points
#         return disc_points_new,call_prices_array_new,marg
#     else:
#         if N == None:
#             return print("Need to provide N")
#         np.random.seed(12)
#         grid = np.random.normal(K,sigma,N-2)# creating a grid concentrated around the target strike
        
#         grid = np.append([0,truncation_range],grid)# adding 0 and the final truncation point to the list
        
#         grid = np.sort(np.abs(grid))# sorting the grid in ascending order
        
#         #next we find the point where the interpolated line joining the last two strike points takes value zero
#         R_0_T1 = find_zero(BS_call,stock,grid,T_GRID[1],sigma,r)
#         print(f"The strike for time {T_GRID[1]} is {R_0_T1} and interpolated call value: {interpolated_line(stock,grid,T_GRID[1],sigma,r,R_0_T1)}")
        
#         R_0_T = find_zero(BS_call,stock,grid,T,sigma,r)
#         print(f"The strike for time {T} is {R_0_T} and interpolated call value: {interpolated_line(stock,grid,T,sigma,r,R_0_T)}")
        
#         R_0 = max(R_0_T1,R_0_T)
        
# #         #next we find the point where the interpolated line joining the last two strike points takes value zero
# #         R_0 = find_zero(BS_call,stock, grid,T,sigma,r)
        
# #         print(f"The interpolated line takes the value 0 at {R_0}")
        
# #         #print the corresponding call value
# #         print(f"Strike:{R_0},Interpolated call value:{interpolated_line(stock,grid,T,sigma,r,R_0)}")
        
#         new_grid = np.append(grid,[R_0])
#         K1_new,K2_new = new_grid,new_grid
        
#         K1_new = np.append(grid[:-1],[R_0_T1])
        
#         if R_0_T1 <= R_0_T:
#             K2_new = np.append(grid[:-1],[R_0_T]) 
#         else:
#             K2_new = np.append(grid,[R_0_T1]) 
        
# #         K1_new, K2_new = np.append(grid,[R_0_T1]),np.append(grid,[R_0_T])
#         #getting the new discretization points
#         disc_points_new = disc_points_new_1(K1_new,K2_new)
#         print(f"The new discretization points are:{disc_points_new}")
#         print(f"Number of discretization points are:{len(K1_new)}")
        
                
#         # store the call prices as a 2D array
#         call_prices_array_new = np.vstack((call_prices_new(stock,grid,K1_new,T_GRID[1],sigma,r),
#                                            call_prices_new(stock,grid,K2_new,T,sigma,r)))
# #         print(call_prices_array_new)
        
#         #getting the discrete marginal distributions

#         marg = marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID)
#         print(marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID))
        
#         #returns the grid and the number of grid points
#         return disc_points_new,call_prices_array_new,marg

In [6]:
# #function to return the discretization points, call prices and the marginals for a given value of truncation range and number of discretization points and choice of uniform or nonuniform grid

# def get_disc_points_and_marginals(truncation_range=None,n=None, N=None,choice="uniform"):
#     """
#        Generate the discretization points for the given problem depending on uniform or non-uniform grid
#        Parameters: 
#           Global variables: Stock, sigma, T,r,
#           Local variables: n- (1/n) denotes the spacing between the discretization points
                        
#     """
#     if choice == "uniform":
#         if truncation_range == None or n == None:
#             return ("Need to provide n or truncation range")
#         #return a uniform grid
#         D = np.linspace(0, truncation_range, int(truncation_range*n)+1)
        
        
#         #next we find the point where the interpolated line joining the last two strike points takes value zero
#         R_0_T1 = find_zero(BS_call,stock,D,T_GRID[1],sigma,r)
#         print(f"The strike for time {T_GRID[1]} is {R_0_T1} and interpolated call value: {interpolated_line(stock,D,T_GRID[1],sigma,r,R_0_T1)}")
        
#         R_0_T = find_zero(BS_call,stock,D,T_GRID[1],sigma,r)
#         print(f"The strike for time {T} is {R_0_T} and interpolated call value: {interpolated_line(stock,D,T,sigma,r,R_0_T)}")
        
#         R_0 = max(R_0_T1,R_0_T)
# #         print(f"The interpolated line takes the value 0 at {R_0}")
       
#         #print the corresponding call value
#         print(f"Strike:{R_0},Interpolated call value:{interpolated_line(stock,D,T_GRID[1],sigma,r,R_0)}")
        
        
#         #extending the truncation range to include the new points till the zero of the interpolated call value
#         K1_new = append_numbers(D,D[-1],R_0,np.diff(D)[0])
        
# #         print(f"Extended truncation range for time {t1[1]}:{K1_new}")
# #         print(f"length of new truncation range :{len(K1_new)}")
# #         print(f"previous length:{len(D)}")
        
#         K2_new = append_numbers(D,D[-1],R_0,np.diff(D)[0])
# #         print(f"Extended truncation range for time {t1[2]}:{K2_new}")
# #         print(f"spacing:{K2_new[-2]/(len(K2_new)-2)},{D[-1]/(len(D)-1)}") 
        
        
#         #getting the new discretization points
#         disc_points_new = disc_points_new_1(K1_new,K2_new)
#         print(f"The new discretization points are:{disc_points_new}")
        
#         # store the call prices as a 2D array
#         call_prices_array_new = np.vstack((call_prices(stock,D,K1_new,T_GRID[1],sigma,r),call_prices(stock,D,K2_new,T,sigma,r)))
# #         print(call_prices_array_new)
        
#         #getting the discrete marginal distributions

#         marg = marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID)
# #         print(marginal_new(call_prices_array_new,disc_points_new,timepoints,t1))
        
        
#         #returns the grid and the number of grid points
#         return disc_points_new,call_prices_array_new,marg
#     else:
#         if N == None:
#             return print("Need to provide N")
#         np.random.seed(12)
#         grid = np.random.normal(K,sigma,N-2)# creating a grid concentrated around the target strike
        
#         grid = np.append([0,truncation_range],grid)# adding 0 and the final truncation point to the list
        
#         grid = np.sort(grid)# sorting the grid in ascending order
        
#         #next we find the point where the interpolated line joining the last two strike points takes value zero
#         R_0_T1 = find_zero(BS_call,stock,grid,T_GRID[1],sigma,r)
#         print(f"The strike for time {T_GRID[1]} is {R_0_T1} and interpolated call value: {interpolated_line(stock,grid,T_GRID[1],sigma,r,R_0_T1)}")
        
#         R_0_T = find_zero(BS_call,stock,grid,T,sigma,r)
#         print(f"The strike for time {T} is {R_0_T} and interpolated call value: {interpolated_line(stock,grid,T,sigma,r,R_0_T)}")
        
#         R_0 = max(R_0_T1,R_0_T)
        
# #         #next we find the point where the interpolated line joining the last two strike points takes value zero
# #         R_0 = find_zero(BS_call,stock, grid,T,sigma,r)
        
# #         print(f"The interpolated line takes the value 0 at {R_0}")
        
# #         #print the corresponding call value
# #         print(f"Strike:{R_0},Interpolated call value:{interpolated_line(stock,grid,T,sigma,r,R_0)}")
        
#         new_grid = np.append(grid,[R_0])
#         K1_new,K2_new = new_grid,new_grid
        
# #         K1_new = np.append(grid[:-1],[R_0_T1])
        
# #         if R_0_T1 <= R_0_T:
# #             K2_new = np.append(grid[:-1],[R_0_T]) 
# #         else:
# #             K2_new = np.append(grid,[R_0_T1]) 
        
# #         K1_new, K2_new = np.append(grid,[R_0_T1]),np.append(grid,[R_0_T])
#         #getting the new discretization points
#         disc_points_new = disc_points_new_1(K1_new,K2_new)
#         print(f"The new discretization points are:{disc_points_new}")
#         print(f"Number of discretization points are:{len(K1_new)}")
        
                
#         # store the call prices as a 2D array
#         call_prices_array_new = np.vstack((call_prices(stock,grid,K1_new,T_GRID[1],sigma,r),
#                                            call_prices(stock,grid,K2_new,T,sigma,r)))
# #         print(call_prices_array_new)
        
#         #getting the discrete marginal distributions

#         marg = marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID)
#         print(marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID))
        
#         #returns the grid and the number of grid points
#         return disc_points_new,call_prices_array_new,marg

In [7]:
#function to return the discretization points, call prices and the marginals for a given value of truncation range and number of discretization points and choice of uniform or nonuniform grid

def get_disc_points_and_marginals(truncation_range=None,n=None, N=None,choice="uniform"):
    """
       Generate the discretization points for the given problem depending on uniform or non-uniform grid
       Parameters: 
          Global variables: Stock, sigma, T,r,
          Local variables: n- (1/n) denotes the spacing between the discretization points
                        
    """
    if choice == "uniform":
        if truncation_range == None or n == None:
            return ("Need to provide n or truncation range")
        #return a uniform grid
        D = np.linspace(0, truncation_range, int(truncation_range*n)+1)
        
        
        #next we find the point where the interpolated line joining the last two strike points takes value zero
        R_0_T1 = find_zero(BS_call,stock,D,T_GRID[1],sigma,r)
        print(f"The strike for time {T_GRID[1]} is {R_0_T1} and interpolated call value: {interpolated_line(stock,D,T_GRID[1],sigma,r,R_0_T1)}")
        
        R_0_T = find_zero(BS_call,stock,D,T_GRID[1],sigma,r)
        print(f"The strike for time {T} is {R_0_T} and interpolated call value: {interpolated_line(stock,D,T,sigma,r,R_0_T)}")
        
        R_0 = max(R_0_T1,R_0_T)
#         print(f"The interpolated line takes the value 0 at {R_0}")
       
        #print the corresponding call value
        print(f"Strike:{R_0},Interpolated call value:{interpolated_line(stock,D,T_GRID[1],sigma,r,R_0)}")
        
        
        #extending the truncation range to include the new points till the zero of the interpolated call value
        K1_new = append_numbers(D,D[-1],R_0,np.diff(D)[0])
        
#         print(f"Extended truncation range for time {t1[1]}:{K1_new}")
#         print(f"length of new truncation range :{len(K1_new)}")
#         print(f"previous length:{len(D)}")
        
        K2_new = append_numbers(D,D[-1],R_0,np.diff(D)[0])
#         print(f"Extended truncation range for time {t1[2]}:{K2_new}")
#         print(f"spacing:{K2_new[-2]/(len(K2_new)-2)},{D[-1]/(len(D)-1)}") 
        
        
        #getting the new discretization points
        disc_points_new = disc_points_new_1(K1_new,K2_new)
        print(f"The new discretization points are:{disc_points_new}")
        
        # store the call prices as a 2D array
        call_prices_array_new = np.vstack((call_prices(stock,D,K1_new,T_GRID[1],sigma,r),call_prices(stock,D,K2_new,T,sigma,r)))
#         print(call_prices_array_new)
        
        #getting the discrete marginal distributions

        marg = marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID)
#         print(marginal_new(call_prices_array_new,disc_points_new,timepoints,t1))
        
        
        #returns the grid and the number of grid points
        return disc_points_new,call_prices_array_new,marg
    else:
        if N == None:
            return print("Need to provide N")
        np.random.seed(12)
        grid = np.random.normal(K,sigma,N-2)# creating a grid concentrated around the target strike
        
        grid = np.append([0,truncation_range],grid)# adding 0 and the final truncation point to the list
        
        grid = np.sort(grid)# sorting the grid in ascending order
        
        #next we find the point where the interpolated line joining the last two strike points takes value zero
        R_0_T1 = find_zero(BS_call,stock,grid,T_GRID[1],sigma,r)
        print(f"The strike for time {T_GRID[1]} is {R_0_T1} and interpolated call value: {interpolated_line(stock,grid,T_GRID[1],sigma,r,R_0_T1)}")
        
        R_0_T = find_zero(BS_call,stock,grid,T,sigma,r)
        print(f"The strike for time {T} is {R_0_T} and interpolated call value: {interpolated_line(stock,grid,T,sigma,r,R_0_T)}")
        
        R_0 = max(R_0_T1,R_0_T)
        
#         #next we find the point where the interpolated line joining the last two strike points takes value zero
#         R_0 = find_zero(BS_call,stock, grid,T,sigma,r)
        
#         print(f"The interpolated line takes the value 0 at {R_0}")
        
#         #print the corresponding call value
#         print(f"Strike:{R_0},Interpolated call value:{interpolated_line(stock,grid,T,sigma,r,R_0)}")
        
        new_grid = np.append(grid,[R_0])
        
        K1_new = np.append(grid[:-1],[R_0_T1])
        
        if R_0_T1 <= R_0_T:
            K2_new = np.append(grid[:-1],[R_0_T]) 
        else:
            K2_new = np.append(grid,[R_0_T1]) 
        
#         K1_new, K2_new = np.append(grid,[R_0_T1]),np.append(grid,[R_0_T])
        #getting the new discretization points
        disc_points_new = disc_points_new_1(K1_new,K2_new)
        print(f"The new discretization points are:{disc_points_new}")
        print(f"Number of discretization points are:{len(K1_new)}")
        
                
        # store the call prices as a 2D array
        call_prices_array_new = np.vstack((call_prices_new(stock,grid,K1_new,T_GRID[1],sigma,r),call_prices_new(stock,grid,K2_new,T,sigma,r)))
#         print(call_prices_array_new)
        
        #getting the discrete marginal distributions

        marg = marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID)
#         print(marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID))
        
        #returns the grid and the number of grid points
        return disc_points_new,call_prices_array_new,marg

In [18]:
#function to return the discretization points, call prices and the marginals for a given value of truncation range and number of discretization points and choice of uniform or nonuniform grid

def get_disc_points_and_marginals(truncation_range=None,n=None, N=None,choice="uniform"):
    """
       Generate the discretization points for the given problem depending on uniform or non-uniform grid
       Parameters: 
          Global variables: Stock, sigma, T,r,
          Local variables: n- (1/n) denotes the spacing between the discretization points
                        
    """
    if choice == "uniform":
        if truncation_range == None or n == None:
            return ("Need to provide n or truncation range")
        #return a uniform grid
        D = np.linspace(0, truncation_range, int(truncation_range*n)+1)
        
        
        #next we find the point where the interpolated line joining the last two strike points takes value zero
        R_0_T1 = find_zero(BS_call,stock,D,T_GRID[1],sigma,r)
        print(f"The strike for time {T_GRID[1]} is {R_0_T1} and interpolated call value: {interpolated_line(stock,D,T_GRID[1],sigma,r,R_0_T1)}")
        
        R_0_T = find_zero(BS_call,stock,D,T_GRID[1],sigma,r)
        print(f"The strike for time {T} is {R_0_T} and interpolated call value: {interpolated_line(stock,D,T,sigma,r,R_0_T)}")
        
        R_0 = max(R_0_T1,R_0_T)
#         print(f"The interpolated line takes the value 0 at {R_0}")
       
        #print the corresponding call value
        print(f"Strike:{R_0},Interpolated call value:{interpolated_line(stock,D,T_GRID[1],sigma,r,R_0)}")
        
        
        #extending the truncation range to include the new points till the zero of the interpolated call value
        K1_new = append_numbers(D,D[-1],R_0,np.diff(D)[0])
        
#         print(f"Extended truncation range for time {t1[1]}:{K1_new}")
#         print(f"length of new truncation range :{len(K1_new)}")
#         print(f"previous length:{len(D)}")
        
        K2_new = append_numbers(D,D[-1],R_0,np.diff(D)[0])
#         print(f"Extended truncation range for time {t1[2]}:{K2_new}")
#         print(f"spacing:{K2_new[-2]/(len(K2_new)-2)},{D[-1]/(len(D)-1)}") 
        
        
        #getting the new discretization points
        disc_points_new = disc_points_new_1(K1_new,K2_new)
        print(f"The new discretization points are:{disc_points_new}")
        
        # store the call prices as a 2D array
        call_prices_array_new = np.vstack((call_prices(stock,D,K1_new,T_GRID[1],sigma,r),call_prices(stock,D,K2_new,T,sigma,r)))
#         print(call_prices_array_new)
        
        #getting the discrete marginal distributions

        marg = marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID)
#         print(marginal_new(call_prices_array_new,disc_points_new,timepoints,t1))
        
        
        #returns the grid and the number of grid points
        return disc_points_new,call_prices_array_new,marg
    else:
        if N == None:
            return print("Need to provide N")
        np.random.seed(12)
        grid = np.random.normal(K,sigma,N-2)# creating a grid concentrated around the target strike
        
        grid = np.append([0,truncation_range],grid)# adding 0 and the final truncation point to the list
        
        grid = np.sort(grid)# sorting the grid in ascending order
        
        #next we find the point where the interpolated line joining the last two strike points takes value zero
        R_0_T1 = find_zero(BS_call,stock,grid,T_GRID[1],sigma,r)
        print(f"The strike for time {T_GRID[1]} is {R_0_T1} and interpolated call value: {interpolated_line(stock,grid,T_GRID[1],sigma,r,R_0_T1)}")
        
        R_0_T = find_zero(BS_call,stock,grid,T,sigma,r)
        print(f"The strike for time {T} is {R_0_T} and interpolated call value: {interpolated_line(stock,grid,T,sigma,r,R_0_T)}")
        
        R_0 = max(R_0_T1,R_0_T)
        
#         #next we find the point where the interpolated line joining the last two strike points takes value zero
#         R_0 = find_zero(BS_call,stock, grid,T,sigma,r)
        
#         print(f"The interpolated line takes the value 0 at {R_0}")
        
#         #print the corresponding call value
#         print(f"Strike:{R_0},Interpolated call value:{interpolated_line(stock,grid,T,sigma,r,R_0)}")
        
        new_grid = np.append(grid,[R_0])
        
        K1_new = np.append(grid[:-1],[R_0_T1])
        
        if R_0_T1 <= R_0_T:
            K2_new = np.append(grid[:-1],[R_0_T]) 
        else:
            K2_new = np.append(grid,[R_0_T1]) 
        
#         K1_new, K2_new = np.append(grid,[R_0_T1]),np.append(grid,[R_0_T])
        #getting the new discretization points
        disc_points_new = disc_points_new_1(K1_new,K2_new)
        print(f"The new discretization points are:{disc_points_new}")
        print(f"Number of discretization points are:{len(K1_new)}")
        
                
        # store the call prices as a 2D array
        call_prices_array_new = np.vstack((call_prices_new(stock,grid,K1_new,T_GRID[1],sigma,r),call_prices_new(stock,grid,K2_new,T,sigma,r)))
#         print(call_prices_array_new)
        
        #getting the discrete marginal distributions
        marg = [BS_marginal_from_calls(C_row, K_row)
        for C_row, K_row in zip(call_prices_array_new, disc_points_new)]
        marg = np.array(marg, dtype=object) 
        
        
#         marg = marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID)
  
#         print(marginal_new(call_prices_array_new,disc_points_new,timepoints,T_GRID))
        
        #returns the grid and the number of grid points
        return disc_points_new,call_prices_array_new,marg

In [9]:
# def get_target_call(marg, disc_points_new,K):
#     """
#     returns the price of the target option and the short-maturity options in the hedge portfolio
#     Parameters:
#        marg: marginal distributions, 
#        disc_points_new: the discretization points 
#     """
#     N = len(disc_points_new[1]) #returns the number of discretization points corresponding to the target maturity
#     target_call = sum(marg[1,i] * np.maximum(disc_points_new[1,i]- K,0) for i in range(N))
#     return target_call
    


# def get_short_call(t1,short_maturity,no_of_options,marg,disc_points_new,short_strikes):
#     """
#     returns the price of the target option and the short-maturity options in the hedge portfolio
#     Parameters:
#       t1: list of the time-points given by [0,short_maturity,target_maturity]
#       short_maturity: maturity of the options in the hedge portfolio
#       no_of_options: numner of options in the hedge portfolio
#       marg: marginal distributions
#       disc_points_new: discretization points
#       short_strikes: strikes of the options in the hedge portfolio
#     """
#     short_call = np.zeros(no_of_options)
    
#     L = t1.index(short_maturity)
    
#     N = len(disc_points_new[0])#returns the number of discretization points corresponding to the short maturity
    
#     #Value of short maturity options at time 0
#     # short_call = \sum_{i}\alpha_i(x_i-short_strike)^+
#     for j in range(no_of_options):
#         short_call[j] = sum(marg[L-1,i] * np.maximum(disc_points_new[L-1,i]-short_strikes[j],0) \
#                             for i in range(N))
#         print("Short call value", str(j),"is:",short_call[j])
#     return short_call





In [10]:
def Asian_option_payoff(stock_price_t1,stock_price_T,strike_price):
    return np.maximum(0.5*(stock_price_t1+stock_price_T)-strike_price,0)

 #### SOLVE THE PROBLEM WITHOUT BIG-M CONSTRAINTS ###

In [11]:
# import gurobipy as gp
# from gurobipy import GRB, quicksum
# import numpy as np

# def solve_max_problem_gurobi_without_M(M_bound,
#     K1, K2,
#     Weights_min_prev, short_strikes,short_call, target_strike,
#     marg, Epsilon
# ):
#     """
#     A version of the 'solve_max_problem_gurobi' function that uses row-specific M_i
#     instead of one global M_bound.
#     """
#     # Create a Gurobi model
#     model = gp.Model("Upper-Bound-Problem-RowSpecificM")
#     model.setParam('OutputFlag', 0)

#     N = len(K1)
#     no_of_options = len(short_strikes)

#     # Decision variables
#     X = model.addVars(N, N, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="X")
#     P_slack = model.addVars(N, lb=0, vtype=GRB.CONTINUOUS, name="P_slack")
#     B = model.addVars(N, vtype=GRB.BINARY, name="B")
# #     Delta = model.addVars(N, lb=0, vtype=GRB.CONTINUOUS, name="Delta")

#     # Objective: Maximize sum of P_slack
#     model.setObjective(quicksum(P_slack[i] for i in range(N)), GRB.MAXIMIZE)

    

    

#     #Calculate the cash position value: it should be the difference between the target option price and the hedge_portfolio value at 0 with only options
   
#     cash = quicksum(X[i, j] * max(K2[j]-target_strike,0)
#                     for j in range(N) for i in range(N))\
#                - np.dot(Weights_min_prev, short_call)
   
     
#     for i in range(N):
#          # The hedging_error expression
#         hedging_error = (quicksum(X[i, j] * max(K2[j] - target_strike, 0) for j in range(N)) \
#                         - marg[0][i] * (
#                             cash
#                             + quicksum(
#                                 Weights_min_prev[l] * max(K1[i] - short_strikes[l], 0)
#                                 for l in range(no_of_options)
#                             )
#                            )
#                         )
#         # -- Absolute value constraints:
#         # a) hedging_error <= P_slack[i]
#         model.addConstr(hedging_error <= P_slack[i], f"modulus_1_{i}")

#         # b) hedging_error >= -P_slack[i]
#         model.addConstr(hedging_error >= -P_slack[i], f"modulus_2_{i}")

#         # d) hedging_error *(1 - 2*B[i]) == P_slack[i]
#         model.addConstr(hedging_error *(1 - 2*B[i]) == P_slack[i], f"modulus_3_{i}")
   
   
#     # Marginal constraints
#     for i in range(N):
#         model.addConstr(quicksum(X[i, j] for j in range(N)) == marg[0][i], f"time1_{i}")

#     for j in range(N):
#         model.addConstr(quicksum(X[i, j] for i in range(N)) == marg[1][j], f"time2_{j}")

#     # Martingale constraints 
   
#     for i in range(N):
#         model.addConstr(quicksum(X[i, j]*(K2[j] - K1[i]) for j in range(N)) == 0, f"martingale_{i}")
    
#     model.setParam('NonConvex', 2)

#     # Solve the model
#     model.optimize()

#     if model.status != GRB.OPTIMAL:
#         return {
#             "call_value": float("inf"),
#             "Status": model.status,
#             "p_matrix": None,
#             "P_slack": None,
#             "B": None,
# #             "Delta": None
#         }
    
 
#     # 6) Extract solution
#     p_matrix = np.zeros((N, N))
#     for i in range(N):
#         for j in range(N):
#             p_matrix[i, j] = X[i, j].X

#     p_slack_array = np.array([P_slack[i].X for i in range(N)])
#     b_array       = np.array([B[i].X          for i in range(N)])
# #     delta_array   = np.array([Delta[i].X      for i in range(N)])
#     call_value    = model.objVal

#     return {
#         "call_value": call_value,
#         "Status": "Optimal",
#         "p_matrix": p_matrix,
#         "P_slack": p_slack_array,
#         "B": b_array,
# #         "Delta": delta_array,
       
#     }


In [12]:
import gurobipy as gp
from gurobipy import GRB, quicksum
import numpy as np

def solve_max_problem_gurobi(
    K1, K2, target_strike,
    marg, Epsilon
):
    """
    A version of the 'solve_max_problem_gurobi' function that uses row-specific M_i
    instead of one global M_bound.
    """
    # Create a Gurobi model
    model = gp.Model("Upper-Bound-Problem-RowSpecificM")
    model.setParam('OutputFlag', 0)

    N_timepoint1 = len(K1)
    N_timepoint2 = len(K2)

    # Decision variables
    X = model.addVars(N_timepoint1, N_timepoint2, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="X")
 


    # Objective: Maximize sum of P_slack
    model.setObjective(quicksum(X[i,j] * Asian_option_payoff(K1[i],K2[j],target_strike)
                                for j in range(N_timepoint2) for i in range(N_timepoint1)), GRB.MAXIMIZE)

    
  
    
    # Marginal constraints
    for i in range(N_timepoint1):
        model.addConstr(quicksum(X[i, j] for j in range(N_timepoint2)) == marg[0][i], f"time1_{i}")

    for j in range(N_timepoint2):
        model.addConstr(quicksum(X[i, j] for i in range(N_timepoint1)) == marg[1][j], f"time2_{j}")

    # Martingale constraints 
   
    for i in range(N_timepoint1):
        model.addConstr(quicksum(X[i, j]*(K2[j] - K1[i]) for j in range(N_timepoint2)) == 0, f"martingale_{i}")
    

    # Solve the model
    model.optimize()

    if model.status != GRB.OPTIMAL:
        return {
            "call_value": float("inf"),
            "Status": model.status,
            "p_matrix": None
        }
    
 
    # 6) Extract solution
    p_matrix = np.zeros((N_timepoint1, N_timepoint2))
    for i in range(N_timepoint1):
        for j in range(N_timepoint2):
            p_matrix[i, j] = X[i, j].X

 
    call_value    = model.objVal

    return {
        "call_value": call_value,
        "Status": "Optimal",
        "p_matrix": p_matrix
    }


In [13]:
# import gurobipy as gp
# from gurobipy import GRB, quicksum
# import numpy as np

# def solve_max_problem_gurobi(
#     K1, K2, target_strike,
#     marg, Epsilon
# ):
#     """
#     A version of the 'solve_max_problem_gurobi' function that uses row-specific M_i
#     instead of one global M_bound.
#     """
#     # Create a Gurobi model
#     model = gp.Model("Upper-Bound-Problem-RowSpecificM")
#     model.setParam('OutputFlag', 0)

#     N = len(K1)
   

#     # Decision variables
#     X = model.addVars(N, N, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="X")
 


#     # Objective: Maximize sum of P_slack
#     model.setObjective(quicksum(X[i,j] * Asian_option_payoff(K1[i],K2[j],target_strike)
#                                 for j in range(N) for i in range(N)), GRB.MAXIMIZE)

    
  
    
#     # Marginal constraints
#     for i in range(N):
#         model.addConstr(quicksum(X[i, j] for j in range(N)) == marg[0][i], f"time1_{i}")

#     for j in range(N):
#         model.addConstr(quicksum(X[i, j] for i in range(N)) == marg[1][j], f"time2_{j}")

#     # Martingale constraints 
   
#     for i in range(N):
#         model.addConstr(quicksum(X[i, j]*(K2[j] - K1[i]) for j in range(N)) == 0, f"martingale_{i}")
    

#     # Solve the model
#     model.optimize()

#     if model.status != GRB.OPTIMAL:
#         return {
#             "call_value": float("inf"),
#             "Status": model.status,
#             "p_matrix": None
#         }
    
 
#     # 6) Extract solution
#     p_matrix = np.zeros((N, N))
#     for i in range(N):
#         for j in range(N):
#             p_matrix[i, j] = X[i, j].X

 
#     call_value    = model.objVal

#     return {
#         "call_value": call_value,
#         "Status": "Optimal",
#         "p_matrix": p_matrix
#     }


In [14]:
###############################################################################
#  MAIN DRIVER (run_experiment) 
###############################################################################
def run_experiment_no_M(solver=None,
    truncation_range=None,
    n=None,
    N=None,
    no_of_options=None,
    epsilon=None,
    M_bound=None,
    choice=None,
    x0=None
):
    """
    Orchestrates everything:
     1) Build discretization points & marginals
     2) Pick short strikes
     3) Compute target call & short calls
     4) Solve min-max
     5) Return final results
    """
    # Build grid & marginals
    disc_points_new, call_prices_array_new, marg = get_disc_points_and_marginals( truncation_range=truncation_range,\
                                                                                n=n,N=N,choice=choice)
    

    K1 = disc_points_new[0]
    
    K2 = disc_points_new[1]
    

    
    # pick solver function
    if solver.lower() == "pulp":
        solver_func = solve_max_problem_pulp
    else:
        solver_func = solve_max_problem_gurobi


   
    # Solve the max problem once more with final_weights
    final_solution = solver_func(
        K1,
        K2,
        K,
        marg,
        epsilon
    )
    
 

#     # Re-compute the target_call
   
#     prob_matrix_final = final_solution["p_matrix"]
#     target_call = 0
#     n_timepoint_1 = len(K1)
#     n_timepoint_2 = len(K2)
    
#     for i in range(n_timepoint_1):
#         for j in range(n_timepoint_2):
#             target_call += prob_matrix_final[i][j]* Asian_option_payoff(K1[i],K2[j],K)
    
#     final_cash = target_call - np.dot(w_opt,short_call)
#     final_weights = np.concatenate(([final_cash], w_opt))
    
#     print("Final full hedge portfolio weights:", final_weights)

#     # 8) Check constraints using the existing check_constraints_pulp_solution function
   
    if final_solution["Status"] =="Optimal" and final_solution["p_matrix"] is not None:
        all_satisfied, violations = check_constraints_pulp_solution_no_M(
            K1,
            K2,
            K,
            marg,
            epsilon,
            final_solution,
            tolerance=1e-08,
        )
    

        if all_satisfied:
            print("All constraints are satisfied at the final solution!")
        else:
            print("Constraint violations found at the final solution:")
            for v in violations:
                print("  -", v)
    else:
        print("Final solution wasn't optimal, so skipping constraint check.")

   
    # 9) Return final results
    return {
        "N_initial": N, #initial choice of discretization points
        "N": len(K1),#total number of discretization points after interpolation 
        "Value_inner":final_solution["call_value"],
        "p_matrix": final_solution["p_matrix"],
        "marg": marg,
        "K1":K1,
        "K2":K2,
    }

In [15]:
# import numpy as np

# def check_constraints_pulp_solution_no_M(
#     M_bound,
#     K1, K2,
#     Weights,         # [cash, w1, w2, ...]
#     short_strikes,
#     target_strike,
#     marg,
#     epsilon,
#     solution_dict,
#     tolerance=1e-8
# ):
#     """
#     Re-checks constraints from solve_max_problem_gurobi_without_M.
#     Returns (all_satisfied, violations).
#     """

#     # ============= 1) Basic Setup =============
#     n_timepoint_1 = len(marg[0])  # alpha
#     n_timepoint_2 = len(marg[1])  # beta
#     no_of_options = len(short_strikes)

#     # Extract final solution arrays
#     p_matrix = solution_dict["p_matrix"]  # shape (n_timepoint_1, n_timepoint_2)
#     P_slack  = solution_dict["P_slack"]   # shape (n_timepoint_1,)
#     B_array  = solution_dict["B"]         # shape (n_timepoint_1,)

#     # If solver not optimal or p_matrix is None -> skip
#     if solution_dict["Status"] != "Optimal" or p_matrix is None:
#         return (
#             False,
#             [f"Solver status {solution_dict['Status']} or p_matrix=None => skip check"]
#         )

#     violations = []

#     # ============= 2) Timepoint 1 Constraints =============
#     # sum_j p[i,j] == marg[0][i]
#     for i in range(n_timepoint_1):
#         lhs = np.sum(p_matrix[i,:])
#         rhs = marg[0][i]
#         if abs(lhs - rhs) > tolerance:
#             msg = f"Row sum constraint violated at i={i}: LHS={lhs}, RHS={rhs}"
#             violations.append(msg)

#     # ============= 3) Timepoint 2 Constraints =============
#     # sum_i p[i,j] == marg[1][j]
#     for j in range(n_timepoint_2):
#         lhs = np.sum(p_matrix[:, j])
#         rhs = marg[1][j]
#         if abs(lhs - rhs) > tolerance:
#             msg = f"Column sum constraint violated at j={j}: LHS={lhs}, RHS={rhs}"
#             violations.append(msg)

#     # ============= 4) Martingale Constraints =============
#     # sum_j p[i,j]*(K2[j] - K1[i]) = 0  for each i
#     for i in range(n_timepoint_1):
#         lhs = 0.0
#         for j in range(n_timepoint_2):
#             lhs += p_matrix[i, j]*(K2[j] - K1[i])
#         if abs(lhs) > tolerance:
#             msg = f"Martingale constraint violated at i={i}: LHS={lhs}"
#             violations.append(msg)

#     # ============= 5) Absolute Value (No Big M) Constraints =============
#     # For each i, define:
#     # hedging_error = sum_j p[i,j]*(K2[j]-target_strike)^+
#     #               - alpha[i]*(cash + sum_l W[l+1]*(K1[i]-short_strikes[l])^+)
#     # 1) hedging_error <= P_slack[i]
#     # 2) hedging_error >= -P_slack[i]
#     # 3) hedging_error*(1 - 2*B[i]) == P_slack[i]

#     alpha = marg[0]
#     cash  = Weights[0]
#     w_opts = Weights[1:]  # the short positions
  

#     for i in range(n_timepoint_1):
#         # payoff_target
#         payoff_target = 0.0
#         for j in range(n_timepoint_2):
#             payoff_target += p_matrix[i,j] * max(K2[j] - target_strike, 0)

#         # payoff_hedge
#         payoff_hedge = alpha[i]*(
#             cash + sum(
#                 w_opts[l] * max(K1[i] - short_strikes[l], 0)
#                 for l in range(no_of_options)
#             )
#         )

#         hedging_error = payoff_target - payoff_hedge

#         # 1) hedging_error <= P_slack[i]
#         if hedging_error - P_slack[i] > tolerance:
#             msg = (f"Absolute-value constraint #1 violated i={i}: "
#                    f"hedging_error={hedging_error}, P_slack={P_slack[i]}")
#             violations.append(msg)

#         # 2) hedging_error >= -P_slack[i]
#         if hedging_error + P_slack[i] < -tolerance:
#             msg = (f"Absolute-value constraint #2 violated i={i}: "
#                    f"hedging_error={hedging_error}, P_slack={P_slack[i]}")
#             violations.append(msg)

#         # 3) hedging_error*(1-2*B[i]) == P_slack[i]
#         lhs_3 = hedging_error*(1 - 2*B_array[i])
#         if abs(lhs_3 - P_slack[i]) > tolerance:
#             msg = (f"Absolute-value constraint #3 violated i={i}: "
#                    f"lhs_3={lhs_3}, P_slack={P_slack[i]}, B={B_array[i]}")
#             violations.append(msg)

#     # ============= Finalize =============
#     all_satisfied = (len(violations) == 0)
#     return (all_satisfied, violations)


In [16]:
import numpy as np

def check_constraints_pulp_solution_no_M(
    K1, K2,
    target_strike,
    marg,
    epsilon,
    solution_dict,
    tolerance=1e-8
):
    """
    Re-checks constraints from solve_max_problem_gurobi_without_M.
    Returns (all_satisfied, violations).
    """

    # ============= 1) Basic Setup =============
    n_timepoint_1 = len(marg[0])  # alpha
    n_timepoint_2 = len(marg[1])  # beta
   

    # Extract final solution arrays
    p_matrix = solution_dict["p_matrix"]  # shape (n_timepoint_1, n_timepoint_2)
 

    # If solver not optimal or p_matrix is None -> skip
    if solution_dict["Status"] != "Optimal" or p_matrix is None:
        return (
            False,
            [f"Solver status {solution_dict['Status']} or p_matrix=None => skip check"]
        )

    violations = []

    # ============= 2) Timepoint 1 Constraints =============
    # sum_j p[i,j] == marg[0][i]
    for i in range(n_timepoint_1):
        lhs = np.sum(p_matrix[i,:])
        rhs = marg[0][i]
        if abs(lhs - rhs) > tolerance:
            msg = f"Row sum constraint violated at i={i}: LHS={lhs}, RHS={rhs}"
            violations.append(msg)

    # ============= 3) Timepoint 2 Constraints =============
    # sum_i p[i,j] == marg[1][j]
    for j in range(n_timepoint_2):
        lhs = np.sum(p_matrix[:, j])
        rhs = marg[1][j]
        if abs(lhs - rhs) > tolerance:
            msg = f"Column sum constraint violated at j={j}: LHS={lhs}, RHS={rhs}"
            violations.append(msg)

    # ============= 4) Martingale Constraints =============
    # sum_j p[i,j]*(K2[j] - K1[i]) = 0  for each i
    for i in range(n_timepoint_1):
        lhs = 0.0
        for j in range(n_timepoint_2):
            lhs += p_matrix[i, j]*(K2[j] - K1[i])
        if abs(lhs) > tolerance:
            msg = f"Martingale constraint violated at i={i}: LHS={lhs}"
            violations.append(msg)

 

    # ============= Finalize =============
    all_satisfied = (len(violations) == 0)
    return (all_satisfied, violations)


**SOLVE THE MAXIMIZATION PROBLEM FOR THE ASIAN OPTION**

In [19]:
if __name__ == "__main__":
    
    result_data_no_M = run_experiment_no_M(
        solver="gurobi",#"pulp"
        truncation_range=2,
        N=1000,
        epsilon= 0,#0.00000001
        M_bound= 0,
        choice="non-uniform"
    )
    
    # Make a similar dictionary
    run_dict_no_M = {
        "Solver": "gurobi",
        "Truncation_range": 2,
        "N_initial": result_data_no_M["N_initial"],
        "N": result_data_no_M["N"],
        "epsilon": 0.0,
         "Final_value": result_data_no_M["Value_inner"],
         "P_matrix": result_data_no_M["p_matrix"]
     }
    print("========== Experiment Complete ==========")
   
    print("Final Value from max problem:", result_data_no_M["Value_inner"])

The strike for time 0.5 is 2.000555448466547 and interpolated call value: 1.0918784085699965e-22
The strike for time 1 is 2.012121251837858 and interpolated call value: 2.405573570202213e-19
The new discretization points are:[[0.         0.25786411 0.36899726 ... 1.60833724 1.63331137 2.00055545]
 [0.         0.25786411 0.36899726 ... 1.60833724 1.63331137 2.01212125]]
Number of discretization points are:1000
All constraints are satisfied at the final solution!
========== Experiment Complete ==========
Final Value from max problem: 0.06775156648664124


In [20]:
if __name__ == "__main__":
    
    result_data_no_M = run_experiment_no_M(
        solver="gurobi",#"pulp"
        truncation_range=2,
        N=1000,
        epsilon= 0,#0.00000001
        M_bound= 0,
        choice="non-uniform"
    )
    
    # Make a similar dictionary
    run_dict_no_M = {
        "Solver": "gurobi",
        "Truncation_range": 2,
        "N_initial": result_data_no_M["N_initial"],
        "N": result_data_no_M["N"],
        "epsilon": 0.0,
         "Final_value": result_data_no_M["Value_inner"],
         "P_matrix": result_data_no_M["p_matrix"]
     }
    print("========== Experiment Complete ==========")
   
    print("Final Value from max problem:", result_data_no_M["Value_inner"])

The strike for time 0.5 is 2.000555448466547 and interpolated call value: 1.0918784085699965e-22
The strike for time 1 is 2.012121251837858 and interpolated call value: 2.405573570202213e-19
The new discretization points are:[[0.         0.25786411 0.36899726 ... 1.60833724 1.63331137 2.00055545]
 [0.         0.25786411 0.36899726 ... 1.60833724 1.63331137 2.01212125]]
Number of discretization points are:1000
timepoint: 0.5
1000
Sum of marginals at 0.5 is: 0.999999999999999
timepoint: 1
1000
Sum of marginals at 1 is: 0.9999999999999992
All constraints are satisfied at the final solution!
========== Experiment Complete ==========
Final Value from max problem: 0.06775156648664124


In [64]:
marg,K1,K2 = result_data_no_M['marg'],result_data_no_M['K1'],result_data_no_M['K2']

In [65]:
print(f"Expected value of stock price at time 0 :{np.dot(marg[0],K1)}")
print(f"Expected value of stock price at time 0 :{np.dot(marg[1],K2)}")

Expected value of stock price at time 0 :1.0
Expected value of stock price at time 0 :1.0


In [20]:
if __name__ == "__main__":
    
    result_data_no_M = run_experiment_no_M(
        solver="gurobi",#"pulp"
        truncation_range=2,
        N=14,
        epsilon= 0,#0.00000001
        M_bound= 0,
        choice="non-uniform"
    )
    
    # Make a similar dictionary
    run_dict_no_M = {
        "Solver": "gurobi",
        "Truncation_range": 2,
        "N_initial": result_data_no_M["N_initial"],
        "N": result_data_no_M["N"],
        "epsilon": 0.0,
         "Final_value": result_data_no_M["Value_inner"],
         "P_matrix": result_data_no_M["p_matrix"]
     }
    print("========== Experiment Complete ==========")
   
    print("Final Value from max problem:", result_data_no_M["Value_inner"])

The strike for time 0.5 is 2.0002422164631994 and interpolated call value: -5.4263048183478615e-21
The strike for time 1 is 2.0081747254138964 and interpolated call value: -1.6601845766184287e-19
The new discretization points are:[[0.         0.65985287 0.69305573 0.83860362 0.86371482 0.88043542
  0.97595447 1.00102542 1.0484879  1.0944914  1.09459717 1.15062857
  1.57436388 2.00024222]
 [0.         0.65985287 0.69305573 0.83860362 0.86371482 0.88043542
  0.97595447 1.00102542 1.0484879  1.0944914  1.09459717 1.15062857
  1.57436388 2.00817473]]
Number of discretization points are:14
Constraint violations found at the final solution:
  - Martingale constraint violated at i=1: LHS=-5.903812472652904e-07
  - Martingale constraint violated at i=13: LHS=5.903812472298454e-07
========== Experiment Complete ==========
Final Value from max problem: 0.06796843220585114


In [21]:
import pickle
with open("result_Asian_MOT.pkl","wb") as f:
    pickle.dump(run_dict_no_M,f)
# results_df.to_pickle("Results_from_MOT.pkl")

In [22]:
def Asian_call_MC(stock, K, t1, t2,sigma,r):
    tau = t1 
    tau2 = t2 - t1
    M = 100000
    S1 = np.zeros((M))
    S2 = np.zeros((M))
    random.seed(10)
    sum1 = 0
    for i in range(M):
        #simulating stock paths at time u
        S1[i] = stock * np.exp((r - 0.5 * sigma**2) * tau + sigma * np.sqrt(tau) * random.gauss(0,1))
        
        #simulating stock paths at time v
        S2[i] = S1[i] * np.exp((r - 0.5 * sigma**2) * tau2 + sigma * np.sqrt(tau2) * random.gauss(0,1))
            
        #payoff for the hedge portfolio with short maturity options 
        #sum1 += multiperiod_payoff(S1[i],S2[i],Strike_1,Strike_2,weight_1,weight_2,K)
        sum1 += Asian_option_payoff( S1[i], S2[i],K)
        #sum1 += max(0.5*(S1[i]+S2[i])-K,0)
    call = np.exp(-r * t2) * (sum1/M)
    return call

In [23]:
print(Asian_call_MC(stock, K, T_GRID[1],T, sigma, r))

0.06284851984627812


In [24]:
print((result_data_no_M["Value_inner"]-Asian_call_MC(stock, K, T_GRID[1],T, sigma, r))/
      Asian_call_MC(stock, K, T_GRID[1],T, sigma, r))

0.08146432679871958


In [ ]:
marg,K1,K2 = result_data_no_M['marg'],result_data_no_M['K1'],result_data_no_M['K2']